In [1]:
import os
import time

import numpy as np

from bng_simulator.utils.services_utils import send_request
# from bng_simulator.utils.io_dict_utils import(
#     save_yaml,
#     round_dict_values,
#     build_tree_from_dict,
#     convert_tree_into_proper_dict
# )

In [2]:
# Assume these are provided by your simulator's API:
def send_control(throttle: float, steering: float, brake: float):
    """
    Command the vehicle with throttle (0.0 to 1.0), steering (-1.0 to 1.0),
    and brake (0.0 to 1.0) values.
    """
    # Implementation specific to the simulator.
    control = {
        "filter": "FILTER_AI",
        "throttle": throttle,
        "steering": steering,
        "brake": brake
    }
    send_request("control_vehicle", control)

# A helper function to reset controls (zero throttle, straight steering, no brake)
def reset_controls():
    send_control(throttle=0.0, steering=0.0, brake=0.0)

In [3]:
# ------------------------------------------------------------------------------
# Experiment 1: Throttle Step Test
# ------------------------------------------------------------------------------
def throttle_step_test(step_throttle: float = 1.0, hold_duration: float = 2.0, rest_duration: float = 2.0):
    """
    From standstill, apply a step change from low throttle to 'step_throttle'
    and hold for a specified duration, then return to zero.
    """
    print("Starting Throttle Step Test")
    
    # Ensure vehicle is at rest and controls are reset
    reset_controls()
    time.sleep(1)  # wait a bit before starting

    # Step 1: Start at low throttle (e.g., 0.1) for a baseline period.
    low_throttle = 0.25
    send_control(throttle=low_throttle, steering=0.0, brake=0.0)
    print("Baseline throttle at {:.2f}".format(low_throttle))
    time.sleep(2)  # hold baseline for 2 seconds

    # Step 2: Step up throttle to step_throttle value.
    send_control(throttle=step_throttle, steering=0.0, brake=0.0)
    print("Stepping throttle to {:.2f}".format(step_throttle))
    time.sleep(hold_duration)

    # Step 3: Step down back to low throttle.
    send_control(throttle=low_throttle, steering=0.0, brake=0.0)
    print("Returning throttle to baseline {:.2f}".format(low_throttle))
    time.sleep(rest_duration)
    
    # Reset controls
    reset_controls()
    print("Throttle Step Test completed.\n")
    time.sleep(1)

# ------------------------------------------------------------------------------
# Experiment 2: Throttle Ramp Test
# ------------------------------------------------------------------------------
def throttle_ramp_test(start_throttle: float = 0.0, end_throttle: float = 1.0, ramp_duration: float = 5.0):
    """
    Smoothly ramp the throttle from start_throttle to end_throttle over ramp_duration.
    """
    print("Starting Throttle Ramp Test")
    
    reset_controls()
    time.sleep(1)

    num_steps = 50  # Number of incremental steps in the ramp
    step_delay = ramp_duration / num_steps

    # Ramp up
    for i in range(num_steps + 1):
        current_throttle = start_throttle + (end_throttle - start_throttle) * (i / num_steps)
        send_control(throttle=current_throttle, steering=0.0, brake=0.0)
        time.sleep(step_delay)
    print("Ramp up complete to throttle {:.2f}".format(end_throttle))
    
    # Hold at end throttle briefly
    time.sleep(2)

    # Ramp down
    for i in range(num_steps + 1):
        current_throttle = end_throttle - (end_throttle - start_throttle) * (i / num_steps)
        send_control(throttle=current_throttle, steering=0.0, brake=0.0)
        time.sleep(step_delay)
    print("Ramp down complete back to throttle {:.2f}".format(start_throttle))
    
    reset_controls()
    print("Throttle Ramp Test completed.\n")
    time.sleep(1)
    

# ------------------------------------------------------------------------------
# Experiment 3: Steady-State Throttle Hold
# ------------------------------------------------------------------------------
def steady_state_test(throttle_value: float = 0.5, hold_duration: float = 5.0):
    """
    Hold a constant throttle value for a specified duration.
    """
    print("Starting Steady-State Throttle Hold Test at throttle {:.2f}".format(throttle_value))
    
    reset_controls()
    time.sleep(1)

    send_control(throttle=throttle_value, steering=0.0, brake=0.0)
    time.sleep(hold_duration)
    
    reset_controls()
    print("Steady-State Test completed.\n")
    time.sleep(1)

# ------------------------------------------------------------------------------
# Experiment 4: Off-Throttle / Engine Braking Test
# ------------------------------------------------------------------------------
def off_throttle_engine_braking_test(baseline_throttle: float = 0.7, hold_duration: float = 3.0):
    """
    Start with a certain throttle value, then cut throttle to zero to observe engine braking.
    """
    print("Starting Off-Throttle (Engine Braking) Test")
    
    reset_controls()
    time.sleep(1)
    
    # Start with a baseline throttle to simulate driving
    send_control(throttle=baseline_throttle, steering=0.0, brake=0.0)
    print("Holding baseline throttle at {:.2f}".format(baseline_throttle))
    time.sleep(hold_duration)
    
    # Cut throttle to zero
    send_control(throttle=0.0, steering=0.0, brake=0.0)
    print("Cutting throttle to 0.0 for engine braking")
    time.sleep(hold_duration)
    
    reset_controls()
    print("Off-Throttle Test completed.\n")
    time.sleep(1)

In [4]:
# Let's turn off all adas
send_request("stop_safety_features")

[]

In [5]:
# Set 2WD and High rangebox
_4wd_args = {
    "mode": "2WD",
    "range_mode": "high" # May not matter for 2WD
}
send_request("set_4WD_mode", _4wd_args)

{}

In [6]:
# Make sure front diffs is unlocked
lock_diffs_args = {
    "diff" : "front",
    "lock" : False
}
send_request("lock_diff", lock_diffs_args)

{}

In [7]:
def pausing_between(delay = 2):
    print(f"Pausing for {delay} seconds")
    time.sleep(delay)

In [8]:
delay = 2
# Let's some of the test
throttle_step_test(step_throttle=0.5, hold_duration=5.0, rest_duration=1.0)
pausing_between(delay)
throttle_step_test(step_throttle=1.0, hold_duration=5.0, rest_duration=1.0)
pausing_between(delay)
throttle_step_test(step_throttle=0.3, hold_duration=5.0, rest_duration=1.0)
pausing_between(delay)
throttle_ramp_test(start_throttle=0.0, end_throttle=1.0, ramp_duration=10.0)
pausing_between(delay)
steady_state_test(throttle_value=0.5, hold_duration=5.0)
pausing_between(delay)
steady_state_test(throttle_value=1.0, hold_duration=5.0)
pausing_between(delay)
steady_state_test(throttle_value=0.7, hold_duration=5.0)
pausing_between(delay)
off_throttle_engine_braking_test(baseline_throttle=0.7, hold_duration=10.0)
pausing_between(delay)
off_throttle_engine_braking_test(baseline_throttle=1.0, hold_duration=5.0)

Starting Throttle Step Test
Baseline throttle at 0.25
Stepping throttle to 0.50
Returning throttle to baseline 0.25
Throttle Step Test completed.

Pausing for 2 seconds
Starting Throttle Step Test
Baseline throttle at 0.25
Stepping throttle to 1.00
Returning throttle to baseline 0.25
Throttle Step Test completed.

Pausing for 2 seconds
Starting Throttle Step Test
Baseline throttle at 0.25
Stepping throttle to 0.30
Returning throttle to baseline 0.25
Throttle Step Test completed.

Pausing for 2 seconds
Starting Throttle Ramp Test
Ramp up complete to throttle 1.00
Ramp down complete back to throttle 0.00
Throttle Ramp Test completed.

Pausing for 2 seconds
Starting Steady-State Throttle Hold Test at throttle 0.50
Steady-State Test completed.

Pausing for 2 seconds
Starting Steady-State Throttle Hold Test at throttle 1.00
Steady-State Test completed.

Pausing for 2 seconds
Starting Steady-State Throttle Hold Test at throttle 0.70
Steady-State Test completed.

Pausing for 2 seconds
Startin

In [ ]:
# Handle engine infos
engine_infos = send_request("get_engine_infos")

In [ ]:
engine_infos